# Озвучка нашим голосом — бесплатно

Читает текст голосом канала. Считает видеокарта Google, MacBook свободен.

**Один раз сверху:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU** → Сохранить.

**Потом:** вставить текст в ячейку 3 и нажать «Среда выполнения» → «Выполнить все».

Первый запуск ставит модель — 3–5 минут. Дальше текст на 2500 знаков озвучивается
примерно за две минуты, готовый файл скачается сам.

In [ ]:
#@title 1. Установка (3-5 минут, один раз за сеанс)
!pip -q install f5-tts soundfile 2>/dev/null || pip -q install git+https://github.com/SWivid/F5-TTS.git soundfile
!git clone -q https://github.com/Rodenom/ozvuchka.git /content/ozv 2>/dev/null; cd /content/ozv && git pull -q
import torch
print('видеокарта:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else
      'НЕ ВЫДАНА — Среда выполнения → Сменить среду выполнения → T4 GPU')

In [ ]:
#@title 2. Настройки

# СКОРОСТЬ ЧТЕНИЯ. Павел 01.10: «бесплатный звучит быстрее платного». Две мерки
# показали обратное (6,1 слога в секунду против 6,8 у ElevenLabs по чистой речи),
# значит дело не в темпе слогов, а в манере: модель читает фразу залпом, без
# микропауз внутри. Меньше 1 — медленнее.
SPEED = 0.93  #@param {type:"number"}

# ДРОБЛЕНИЕ ДЛИННЫХ ФРАЗ, знаков. Живой человек внутри длинной мысли всё равно
# берёт воздух, синтез — нет. Чем меньше число, тем чаще дыхание внутри
# предложения.
CHUNK = 120  #@param {type:"number"}

ИМЯ_ФАЙЛА = "выпуск"  #@param {type:"string"}
print('скорость %.2f, дробление по %d знаков' % (SPEED, CHUNK))

In [ ]:
#@title 3. Текст выпуска — вставить сюда
ТЕКСТ = """
Two months before the explosion, BP paid a consulting firm to walk through this refinery
and write down what the workers said about it. The report came back in January. It described
broken alarms, thinned pipe, chunks of concrete falling, bolts dropping sixty feet, people
getting sick from fumes. That isn't a whistleblower's letter. It's a report the company
commissioned itself, paid for, and filed, eight weeks before the explosion.
"""
print('знаков:', len(ТЕКСТ.strip()))

In [ ]:
#@title 4. Озвучка
# Оживление: фразы отдельно, неровные паузы, плавающий темп и громкость. Ровная
# речь — главное, по чему узнают синтез: замер против живых чтецов дал разброс
# пауз 0,27-0,49 у людей против 0,07 у синтеза одним куском.
import hashlib, random, re, subprocess, os, time
import numpy as np, soundfile as sf
from f5_tts.api import F5TTS

END, MID = '.!?', ',;:'
GAPS = {'.': (0.16,0.75), '!': (0.20,0.80), '?': (0.20,0.80), ':': (0.18,0.42),
        ';': (0.18,0.42), ',': (0.06,0.15), '': (0.12,0.30)}
GAP_K, RUN_ON, LONG_EVERY = 1.15, 0.25, 8

def phrases(text, limit):
    out = []
    for sent in re.findall(r'[^%s]+[%s]*' % (re.escape(END), re.escape(END)), text.replace('\n',' ')):
        sent = sent.strip()
        if not sent: continue
        mark = sent[-1] if sent[-1] in END else ''
        if len(sent) > limit:
            buf = ''
            for p in re.split(r'(?<=[%s])\s*' % re.escape(MID), sent):
                if len(buf)+len(p) > limit and buf:
                    out.append((buf.strip(), ',')); buf = p
                else: buf += ' ' + p
            if buf.strip(): out.append((buf.strip(), mark))
        else: out.append((sent, mark))
    return out

REF_WAV = '/content/ozv/sample/образец.wav'
REF_TXT = open('/content/ozv/sample/образец.txt', encoding='utf-8').read().strip()
tts = F5TTS()
text = ТЕКСТ.strip()
ph = phrases(text, CHUNK)
rnd  = random.Random(hashlib.md5(text.encode()).hexdigest())
vrnd = random.Random(hashlib.md5((text+'vol').encode()).hexdigest())
t0, parts, sr = time.time(), [], 24000
for i,(sent,mark) in enumerate(ph):
    wav, sr, _ = tts.infer(ref_file=REF_WAV, ref_text=REF_TXT, gen_text=sent, remove_silence=False)
    a = np.asarray(wav, dtype=np.float32)
    a = a * (10 ** ((vrnd.uniform(-2,2) - 1.2*(i/max(len(ph)-1,1))) / 20.0))
    parts.append(a)
    if i < len(ph)-1:
        lo,hi = GAPS.get(mark, GAPS[''])
        gap = rnd.uniform(lo,hi)*GAP_K
        if mark in END and i and i % LONG_EVERY == 0: gap = rnd.uniform(0.95,1.45)
        elif rnd.random() < RUN_ON: gap = rnd.uniform(0.05,0.13)
        parts.append(np.zeros(int(gap*sr), np.float32))
    if (i+1) % 10 == 0: print('  %d из %d фраз' % (i+1, len(ph)))
sf.write('/content/raw.wav', np.concatenate(parts), sr)
# дикторский тембр: грудь на 110 Гц, минус бубнёж на 400, дикция на 3,2 кГц,
# громкость как у остальных наших дорожек. Компрессора нет — он съедает динамику.
out = f'/content/{ИМЯ_ФАЙЛА}.mp3'
subprocess.run(['ffmpeg','-y','-v','error','-i','/content/raw.wav','-af',
                f'atempo={SPEED},equalizer=f=110:t=q:w=0.9:g=3.5,'
                'equalizer=f=400:t=q:w=1.0:g=-2.5,equalizer=f=3200:t=q:w=1.2:g=2.0,'
                'loudnorm=I=-19:TP=-1.5:LRA=11', '-b:a','192k', out], check=True)
print('\nготово: %d фраз, %.0f секунд работы' % (len(ph), time.time()-t0))
from google.colab import files
files.download(out)